In [ ]:
#@title Install required libraries
!pip install -q google-generativeai


In [ ]:
#@title Import libraries and setup
import os
import json
import google.generativeai as genai

# Configure API key: Colab userdata first, then environment variable
api_key = None
try:
    from google.colab import userdata  # type: ignore
    api_key = userdata.get("GOOGLE_API_KEY")
except Exception:
    api_key = os.environ.get("GOOGLE_API_KEY")

if not api_key:
    raise RuntimeError(
        "Missing GOOGLE_API_KEY. Set it in Colab userdata or as an environment variable."
    )

# Initialize Gemini client
genai.configure(api_key=api_key)
model = genai.GenerativeModel("gemini-2.5-pro")


In [ ]:
#@title Data Loading Functions
def load_kg_data(file_path):
    """Load Knowledge Graph JSON file as raw string"""
    try:
        with open(file_path, 'r') as f:
            return json.dumps(json.load(f)), None
    except Exception as e:
        return None, f"Error loading KG file: {str(e)}"


def load_raw_data(file_path):
    """Load raw text file as string"""
    try:
        with open(file_path, 'r') as f:
            return f.read(), None
    except Exception as e:
        return None, f"Error loading raw file: {str(e)}"


In [ ]:
#@title Prompt Templates
KG_PROMPT = """
1. Focus on the question:
Analyze the provided TraceCompass State System knowledge graph data to answer the question. You must focus on answering this specific question, and your response should be derived exclusively from the provided knowledge graph data.

2. Use the provided data:
The data you will use is from the TraceCompass State System, specifically in the form of a knowledge graph. Ensure your analysis strictly uses this data for the answer.

3. Leverage your knowledge:
You may use your understanding of TraceCompass State System analysis methods (such as CPU usage analysis, active thread analysis, and Ease script queries) as context for interpreting the data. Keep in mind that the provided data comes from an Ease script query run on the TraceCompass system.

4. Reason over the graph:
The data may not always be explicitly present in the graph, and you may need to reason over the graph's structure and relationships to derive the answer. Use your knowledge of node relationships, temporal patterns, and resource utilization to infer the missing information where necessary.

5. Answer directly:
Provide a direct, concise answer to the question. This answer should be based solely on the provided data.

6. Explain your reasoning:
After providing the direct answer, explain how you arrived at it. Your explanation should cover the following:
   - How the nodes and their relationships helped answer the question.
   - Any specific behaviors or insights derived from the TraceCompass State System analysis, such as CPU usage or thread activity analysis.

7. Question and context:
Question: {question}
TraceCompass Knowledge Graph Context:
{context}

"""

RAW_PROMPT = """
1. Focus on the question:
Analyze the provided raw TraceCompass system trace data to answer the question. Your response must be directly based on this data.

2. Use the provided data:
The data you will use is from the TraceCompass State System in the form of raw system trace measurements. Your analysis should rely exclusively on this data for your answer.

3. Leverage your knowledge:
You may use your understanding of TraceCompass State System analysis methods (such as CPU usage analysis, active thread analysis, and Ease script queries) to interpret the trace data. The provided data is the result of an Ease script query run on the TraceCompass system.

4. Answer directly:
Provide a direct, concise answer to the question based on the raw trace data.

5. Explain your reasoning:
After providing the direct answer, explain how the raw data supports your answer. In your explanation, you should cover:
   - Any patterns or anomalies you identified in the trace data.
   - How these patterns or anomalies are relevant to the TraceCompass system's performance or behavior, particularly related to CPU usage, thread activity, or any Ease script queries.
   - Any performance characteristics or insights drawn from the data, emphasizing the specific context of TraceCompass' analysis.

6. Question and context:
Question: {question}
Raw Data Context:
{context}


"""


In [ ]:
#@title Analysis Functions
def query_gemini(prompt: str) -> str:
    """Execute Gemini 2.5 Pro query."""
    try:
        response = model.generate_content(prompt)
        # Prefer .text if available
        if hasattr(response, "text") and response.text:
            return response.text.strip()
        return str(response).strip()
    except Exception as e:
        return f"API Error: {str(e)}"


def analyze_kg(question: str, file_path: str) -> str:
    """Knowledge Graph analysis pipeline"""
    context, error = load_kg_data(file_path)
    if error:
        return error

    prompt = KG_PROMPT.format(
        question=question,
        context=context  # Context window management
    )
    return query_gemini(prompt)


def analyze_raw(question: str, file_path: str) -> str:
    """Raw data analysis pipeline"""
    context, error = load_raw_data(file_path)
    if error:
        return error

    prompt = RAW_PROMPT.format(
        question=question,
        context=context  # Keep within token limits
    )
    return query_gemini(prompt)


In [ ]:
#@title Main Execution
# def full_analysis(question: str, kg_path: str = "data/cpu_usage_graph.json", raw_path: str = "data/cpu_usage_input.txt"):
#     """Run complete analysis with both approaches"""
#     print(f"\n🔍 Question: {question}")
#     print("="*60)
# 
#     print("\n\n📈 Raw Data Analysis:")
#     print(analyze_raw(question, raw_path))
# 
#     print("\n📊 Knowledge Graph Analysis:")
#     print(analyze_kg(question, kg_path))


# Example usage: test only analyze_kg
if __name__ == "__main__":
    sample_question = "What is the total accumulated CPU time for thread 5130 on CPU 2?"  #@param {type:"string"}
    kg_path = "data/cpu_usage_graph.json"
    print(f"\n🔍 Question: {sample_question}")
    print("="*60)
    print("\n📊 Knowledge Graph Analysis:")
    print(analyze_kg(sample_question, kg_path))
